# Exploratory Data Analysis

Load and inspect the time series dataset.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pathlib import Path
import subprocess
from contextlib import contextmanager
import pandas as pd

# Каталог с файлами Kaggle (можно переопределить)
data_root = Path('/content/drive/MyDrive/favorita-grocery-sales-forecasting (Unzipped Files)')
sample_rows = 100_000

data_root

PosixPath('/content/drive/MyDrive/favorita-grocery-sales-forecasting (Unzipped Files)')

In [4]:
EXPECTED_TABLES = [
    'train.csv',
    'test.csv',
    'stores.csv',
    'items.csv',
    'oil.csv',
    'holidays_events.csv',
    'transactions.csv',
    'sample_submission.csv',
]

REQUIRED_COLUMNS = {
    'train.csv': {'id', 'date', 'store_nbr', 'item_nbr', 'unit_sales', 'onpromotion'},
    'test.csv': {'id', 'date', 'store_nbr', 'item_nbr', 'onpromotion'},
    'stores.csv': {'store_nbr', 'city', 'state', 'type', 'cluster'},
    'items.csv': {'item_nbr', 'family', 'class', 'perishable'},
    'oil.csv': {'date', 'dcoilwtico'},
    'holidays_events.csv': {'date', 'type', 'locale', 'locale_name', 'description', 'transferred'},
    'transactions.csv': {'date', 'store_nbr', 'transactions'},
    'sample_submission.csv': {'id', 'unit_sales'},
}

KEY_COLUMNS = {
    'train.csv': ['id'],
    'test.csv': ['id'],
    'stores.csv': ['store_nbr'],
    'items.csv': ['item_nbr'],
    'transactions.csv': ['date', 'store_nbr'],
    'sample_submission.csv': ['id'],
}

DATE_COLUMNS = {
    'train.csv': ['date'],
    'test.csv': ['date'],
    'oil.csv': ['date'],
    'holidays_events.csv': ['date'],
    'transactions.csv': ['date'],
}

In [5]:
def normalize_table_name(path: Path) -> str:
    name = path.name
    return name[:-3] if name.endswith('.7z') else name

def discover_files(root: Path) -> dict:
    files = {}
    for p in root.rglob('*'):
        if not p.is_file():
            continue
        if p.name.endswith('.csv') or p.name.endswith('.csv.7z'):
            table = normalize_table_name(p)
            files.setdefault(table, []).append(p)
    return files

@contextmanager
def csv_stream(path: Path):
    """Открывает поток CSV как для .csv, так и для .csv.7z (через 7z -so)."""
    proc = None
    fh = None
    try:
        if path.name.endswith('.7z'):
            proc = subprocess.Popen(
                ['7z', 'x', '-so', str(path)],
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
            )
            yield proc.stdout
        else:
            fh = open(path, 'rb')
            yield fh
    finally:
        if fh is not None:
            fh.close()
        if proc is not None:
            if proc.poll() is None:
                proc.terminate()
            proc.wait(timeout=10)

In [6]:
def validate_table(table_name: str, path: Path, sample_rows: int = 100_000) -> tuple:
    result = {
        'table': table_name,
        'source_file': str(path),
        'status': 'ok',
        'sample_rows': 0,
        'sample_columns': 0,
        'missing_required_columns': '',
        'duplicate_keys_in_sample': None,
        'date_parse_error_rate': None,
        'null_pct_top5': '',
    }
    issues = []

    try:
        with csv_stream(path) as stream:
            df = pd.read_csv(stream, nrows=sample_rows, low_memory=False)
    except Exception as e:
        result['status'] = 'read_error'
        issues.append(f'Не удалось прочитать {path.name}: {e}')
        return result, issues

    result['sample_rows'] = len(df)
    result['sample_columns'] = len(df.columns)

    if df.empty:
        result['status'] = 'empty_sample'
        issues.append(f'{table_name}: пустой sample при чтении')

    required = REQUIRED_COLUMNS.get(table_name, set())
    missing = sorted(required - set(df.columns))
    if missing:
        result['status'] = 'schema_issue'
        result['missing_required_columns'] = ', '.join(missing)
        issues.append(f'{table_name}: отсутствуют обязательные колонки: {missing}')

    key_cols = KEY_COLUMNS.get(table_name, [])
    if key_cols and all(col in df.columns for col in key_cols):
        dup_count = int(df.duplicated(subset=key_cols).sum())
        result['duplicate_keys_in_sample'] = dup_count
        if dup_count > 0:
            result['status'] = 'quality_issue' if result['status'] == 'ok' else result['status']
            issues.append(f'{table_name}: дублей по ключу {key_cols} в sample: {dup_count}')

    date_cols = DATE_COLUMNS.get(table_name, [])
    parse_errors = []
    for col in date_cols:
        if col in df.columns:
            parsed = pd.to_datetime(df[col], errors='coerce')
            err_rate = float(parsed.isna().mean())
            parse_errors.append(err_rate)
            if err_rate > 0.0:
                result['status'] = 'quality_issue' if result['status'] == 'ok' else result['status']
                issues.append(f'{table_name}: ошибки парсинга дат в {col}: {err_rate:.2%}')
    if parse_errors:
        result['date_parse_error_rate'] = max(parse_errors)

    if len(df.columns) > 0:
        null_pct = (df.isna().mean() * 100).sort_values(ascending=False).head(5)
        result['null_pct_top5'] = '; '.join([f'{k}={v:.2f}%' for k, v in null_pct.items()])

    return result, issues

In [7]:
found_files = discover_files(data_root)
rows = []
all_issues = []

for table in EXPECTED_TABLES:
    candidates = found_files.get(table, [])
    if not candidates:
        rows.append({
            'table': table,
            'source_file': '',
            'status': 'missing_file',
            'sample_rows': 0,
            'sample_columns': 0,
            'missing_required_columns': '',
            'duplicate_keys_in_sample': None,
            'date_parse_error_rate': None,
            'null_pct_top5': '',
        })
        all_issues.append(f'{table}: файл не найден (ни .csv, ни .csv.7z)')
        continue

    best = sorted(candidates, key=lambda p: (p.name.endswith('.7z'), len(str(p))))[0]
    result, issues = validate_table(table, best, sample_rows=sample_rows)
    rows.append(result)
    all_issues.extend(issues)

report_df = pd.DataFrame(rows).sort_values(['status', 'table']).reset_index(drop=True)

print(f'Корневая папка: {data_root}')
print(f'Найдено таблиц: {len([r for r in rows if r["status"] != "missing_file"])} / {len(EXPECTED_TABLES)}')
display(report_df)

if all_issues:
    print('\nНайденные проблемы:')
    for i, issue in enumerate(all_issues, 1):
        print(f'{i}. {issue}')
else:
    print('\nПроблем не обнаружено в проверяемом sample.')

Корневая папка: /content/drive/MyDrive/favorita-grocery-sales-forecasting (Unzipped Files)
Найдено таблиц: 8 / 8


,table,source_file,status,sample_rows,sample_columns,missing_required_columns,duplicate_keys_in_sample,date_parse_error_rate,null_pct_top5
0,holidays_events.csv,/content/drive/MyDrive/favorita-grocery-sales-...,ok,350,6,,NaN,0.0,date=0.00%; type=0.00%; locale=0.00%; locale_n...
1,items.csv,/content/drive/MyDrive/favorita-grocery-sales-...,ok,4100,4,,0.0,NaN,item_nbr=0.00%; family=0.00%; class=0.00%; per...
2,oil.csv,/content/drive/MyDrive/favorita-grocery-sales-...,ok,1218,2,,NaN,0.0,dcoilwtico=3.53%; date=0.00%
3,sample_submission.csv,/content/drive/MyDrive/favorita-grocery-sales-...,ok,100000,2,,0.0,NaN,id=0.00%; unit_sales=0.00%
4,stores.csv,/content/drive/MyDrive/favorita-grocery-sales-...,ok,54,5,,0.0,NaN,store_nbr=0.00%; city=0.00%; state=0.00%; type...
5,test.csv,/content/drive/MyDrive/favorita-grocery-sales-...,ok,100000,5,,0.0,0.0,id=0.00%; date=0.00%; store_nbr=0.00%; item_nb...
6,train.csv,/content/drive/MyDrive/favorita-grocery-sales-...,ok,100000,6,,0.0,0.0,onpromotion=100.00%; id=0.00%; date=0.00%; sto...
7,transactions.csv,/content/drive/MyDrive/favorita-grocery-sales-...,ok,83488,3,,0.0,0.0,date=0.00%; store_nbr=0.00%; transactions=0.00%



Проблем не обнаружено в проверяемом sample.
